# Per-expert separability — do forget & retain separate *inside* each expert?

Loads the per-expert run (`silhouette_expert.json` + `expert_scatter.npz`). CASAL Fig-19 style: for each (layer, expert), the tokens routed there, PCA'd, forget=red / retain=blue.

**Watch for the format confound:** forget=wmdp (free-text), retain=mmlu (MCQ with options). If the red/blue split looks like it's driven by structure/length rather than being interleaved-then-separated, the "separability" may be *format*, not biology content. The symmetric-MCQ control run is what settles it.

In [ ]:
import json
from pathlib import Path
import numpy as np
import numpy.ma as ma
import matplotlib.pyplot as plt

R = Path('.').resolve().parent / 'run_results/separability/Qwen3-30B-A3B'
d = json.load(open(R / 'silhouette_expert.json'))
S = np.array([[np.nan if v is None else v for v in row] for row in d['silhouette_expert']])  # [L, E]
z = np.load(R / 'expert_scatter.npz')
coords, routing, labels = z['coords'], z['routing'], z['labels']   # [Nt,L,2], [Nt,L,k], [Nt]
L, E = S.shape
print(f'layers={L}  experts={E}  tokens={len(labels)}  top_k={d["top_k"]}')
print(f'cells scored: {np.isfinite(S).sum()} / {S.size}   mean {np.nanmean(S):.3f}   max {np.nanmax(S):.3f}')

## 1. Heatmap — which (layer, expert) cells separate?

Bright = forget/retain separate inside that expert. Grey = too few tokens routed there (`<min_pts`).

In [ ]:
Sm = ma.masked_invalid(S)
cmap = plt.cm.viridis.copy(); cmap.set_bad('lightgrey')
plt.figure(figsize=(13,5))
plt.imshow(Sm, aspect='auto', cmap=cmap, vmin=0, vmax=np.nanmax(S))
plt.colorbar(label='per-expert silhouette (forget vs retain)')
plt.xlabel('expert'); plt.ylabel('layer')
plt.title('Per-expert separability  (bright = separates; grey = <min_pts)')
plt.tight_layout(); plt.show()

## 2. Mean separability by layer

In [ ]:
lm = np.nanmean(S, axis=1)
plt.figure(figsize=(11,3.5))
plt.plot(lm, marker='.')
plt.xlabel('layer'); plt.ylabel('mean per-expert silhouette')
plt.title('Mean per-expert separability by layer'); plt.grid(alpha=.3); plt.tight_layout(); plt.show()
print('top layers:', [(int(l), round(float(lm[l]),2)) for l in np.argsort(-np.nan_to_num(lm,nan=-1))[:6]])

## 3. Inspect one expert — look at the cluster

`show_expert(layer, expert)` plots that expert's tokens. Try the most-separable one vs a routing hazardous-exclusive one (e.g. L34/e69) and see if they look different.

In [ ]:
def show_expert(l, e):
    idx = np.where((routing[:, l, :] == e).any(axis=1))[0]
    if len(idx) == 0:
        print(f'no tokens routed to L{l} e{e}'); return
    Z = coords[idx, l, :].astype(float); y = labels[idx]
    plt.figure(figsize=(5.5,5))
    plt.scatter(Z[y==1,0], Z[y==1,1], s=14, c='#cc4444', alpha=.6, label='forget (wmdp)')
    plt.scatter(Z[y==0,0], Z[y==0,1], s=14, c='#4488aa', alpha=.6, label='retain (mmlu)')
    s = S[l][e]
    plt.title(f'L{l} e{e}   n={len(idx)}   silhouette={s:.2f}' if s==s else f'L{l} e{e}   n={len(idx)}')
    plt.xlabel('PC1'); plt.ylabel('PC2'); plt.legend(); plt.tight_layout(); plt.show()

li, ei = np.unravel_index(np.nanargmax(S), S.shape)
show_expert(int(li), int(ei))      # most separable
show_expert(34, 69)                # a routing hazardous-exclusive expert, for contrast

## 4. Grid of the most-separable experts

In [ ]:
flat = np.argsort(-np.nan_to_num(S.ravel(), nan=-1))[:8]
cells = [np.unravel_index(int(i), S.shape) for i in flat]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, (l, e) in zip(axes.ravel(), cells):
    idx = np.where((routing[:, l, :] == e).any(axis=1))[0]
    Z = coords[idx, l, :].astype(float); y = labels[idx]
    ax.scatter(Z[y==1,0], Z[y==1,1], s=8, c='#cc4444', alpha=.6)
    ax.scatter(Z[y==0,0], Z[y==0,1], s=8, c='#4488aa', alpha=.6)
    ax.set_title(f'L{l} e{e}  sil={S[l][e]:.2f}'); ax.set_xticks([]); ax.set_yticks([])
fig.suptitle('Most-separable experts — forget (red) vs retain (blue)')
plt.tight_layout(); plt.show()